# Model Audit

In [1]:
import time
from pathlib import Path

import torch

from edu_eval.generation.config import GenerationConfig
from edu_eval.generation.runner import GenerationRunner
from edu_eval.models.loader import ModelLoader
from edu_eval.models.registry import ModelRegistry

In [2]:
PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
    and (path / "configs" / "models.yaml").exists()
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "models.yaml"
GENERATION_CONFIG_PATH = PROJECT_ROOT / "configs" / "generation.yaml"

print("Project root :", PROJECT_ROOT)
print("Model config :", CONFIG_PATH)
print("Generation   :", GENERATION_CONFIG_PATH)

Project root : /home/berdangry/Documents/edu-llm-evaluation
Model config : /home/berdangry/Documents/edu-llm-evaluation/configs/models.yaml
Generation   : /home/berdangry/Documents/edu-llm-evaluation/configs/generation.yaml


In [3]:
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

PyTorch : 2.12.1+cu129
CUDA    : 12.9
Available: True
GPU     : NVIDIA GeForce RTX 3050 4GB Laptop GPU


In [4]:
registry = ModelRegistry.from_yaml(CONFIG_PATH)

print("Registered models:")
for spec in registry.all():
    print(f"- {spec.model_id}: {spec.source}")

Registered models:
- smoke_qwen: Qwen/Qwen2.5-0.5B-Instruct


In [5]:
model_id = "smoke_qwen"
model_spec = registry.get(model_id)

print("Model ID :", model_spec.model_id)
print("Role     :", model_spec.model_role)
print("Stage    :", model_spec.stage)
print("Source   :", model_spec.source)

Model ID : smoke_qwen
Role     : smoke_test
Stage    : smoke
Source   : Qwen/Qwen2.5-0.5B-Instruct


In [6]:
print("Loading model...")

start = time.perf_counter()

tokenizer, model, device = ModelLoader.load(model_spec)

load_time = time.perf_counter() - start

print("Done.")
print("Device     :", device)
print("Model      :", type(model).__name__)
print("Dtype      :", next(model.parameters()).dtype)
print("Param dev  :", next(model.parameters()).device)
print("Load time  :", f"{load_time:.2f}s")

Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Done.
Device     : cuda
Model      : Qwen2ForCausalLM
Dtype      : torch.bfloat16
Param dev  : cuda:0
Load time  : 3.32s


In [7]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU allocated : {allocated:.2f} GB")
    print(f"GPU reserved  : {reserved:.2f} GB")

GPU allocated : 0.92 GB
GPU reserved  : 0.93 GB


In [8]:
from edu_eval.models.audit import inspect_model

runtime_info = inspect_model(model, tokenizer)

print("Architecture :", runtime_info.architecture)
print("Model type   :", runtime_info.model_type)
print("Architectures:", runtime_info.architectures)
print("Parameters   :", runtime_info.parameter_count)
print("Dtype        :", runtime_info.dtype)
print("Device       :", runtime_info.device)
print("Context      :", runtime_info.context_length)
print("Vocab size   :", runtime_info.vocab_size)
print("Chat template:", runtime_info.chat_template_available)

Architecture : Qwen2ForCausalLM
Model type   : qwen2
Architectures: ['Qwen2ForCausalLM']
Parameters   : 494032768
Dtype        : bfloat16
Device       : cuda:0
Context      : 32768
Vocab size   : 151936
Chat template: True


In [9]:
has_chat_template = tokenizer.chat_template is not None

print("Chat template available:", has_chat_template)

if has_chat_template:
    print(tokenizer.chat_template[:500])

Chat template available: True
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool 


In [10]:
generation_config = GenerationConfig.from_yaml(
    GENERATION_CONFIG_PATH
)

print(generation_config)

max_new_tokens=256 do_sample=False use_cache=True prompt_version='cg_v1' target_grades=['SD6', 'SMP7', 'SMP9', 'SMA10'] subjects=['IPA', 'Bahasa Indonesia', 'Pendidikan Pancasila', 'Informatika', 'IPS'] pilot_cases_per_subject=2 full_cases_per_subject=12


In [11]:
runner = GenerationRunner(
    model=model,
    tokenizer=tokenizer,
    device=device,
    config=generation_config,
)

print("GenerationRunner ready.")

GenerationRunner ready.


In [12]:
prompt = (
    "Jelaskan fotosintesis untuk siswa kelas 6 SD "
    "dalam bahasa Indonesia."
)

result = runner.generate(prompt)

print("OUTPUT:")
print(result["text"])

print()
print("METADATA:")
print("Input tokens :", result["input_tokens"])
print("Output tokens:", result["output_tokens"])
print("Latency      :", f'{result["latency_seconds"]:.2f}s')
print("Hit ceiling  :", result["hit_max_new_tokens"])

OUTPUT:
Fotosynthesis adalah proses yang digunakan oleh manusia untuk menghasilkan energi dari sinar matahari. Ini adalah cara yang efektif untuk memanfaatkan sinar matahari untuk keperluan hidup manusia.

1. Proses Fotopsik: 
   - Seringkali, manusia menggunakan fotopsik untuk menangkap sinar matahari.
   - Fotopsik ini terdiri daripada dua bagian utama:
     - Fotoplasma: bagian paling dekat dengan matahari dan dapat membantu melindungi matahari.
     - Fotoperas: bagian di sekitarnya yang berfungsi untuk memanfaatkan sinar matahari.

2. Penggunaan Fotopsik:
   - Fotopsik biasanya digunakan untuk:
     - Menyimpan energi: seperti elektron atau potensi energi.
     - Memanfaatkan energi: seperti motor atau mesin pembakaran.
     - Mengubah energi menjadi informasi: seperti penggabungan data.

3. Efek Fotopsik:
   - Fotopsik dapat menyumbang kepada banyak aspek kehidupan manusia, term

METADATA:
Input tokens : 48
Output tokens: 256
Latency      : 4.28s
Hit ceiling  : True


In [13]:
result

{'text': 'Fotosynthesis adalah proses yang digunakan oleh manusia untuk menghasilkan energi dari sinar matahari. Ini adalah cara yang efektif untuk memanfaatkan sinar matahari untuk keperluan hidup manusia.\n\n1. Proses Fotopsik: \n   - Seringkali, manusia menggunakan fotopsik untuk menangkap sinar matahari.\n   - Fotopsik ini terdiri daripada dua bagian utama:\n     - Fotoplasma: bagian paling dekat dengan matahari dan dapat membantu melindungi matahari.\n     - Fotoperas: bagian di sekitarnya yang berfungsi untuk memanfaatkan sinar matahari.\n\n2. Penggunaan Fotopsik:\n   - Fotopsik biasanya digunakan untuk:\n     - Menyimpan energi: seperti elektron atau potensi energi.\n     - Memanfaatkan energi: seperti motor atau mesin pembakaran.\n     - Mengubah energi menjadi informasi: seperti penggabungan data.\n\n3. Efek Fotopsik:\n   - Fotopsik dapat menyumbang kepada banyak aspek kehidupan manusia, term',
 'input_tokens': 48,
 'output_tokens': 256,
 'latency_seconds': 4.283289572000285,


In [14]:
del runner
del model
del tokenizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model resources released.")

Model resources released.
